## 환경 설정하기

In [ ]:
!pip install  -U -q git+https://github.com/huggingface/transformers.git git+https://github.com/huggingface/trl.git datasets bitsandbytes peft qwen-vl-utils wandb accelerate
# Tested with transformers==4.47.0.dev0, trl==0.12.0.dev0, datasets==3.0.2, bitsandbytes==0.44.1, peft==0.13.2, qwen-vl-utils==0.0.8, wandb==0.18.5, accelerate==1.0.1
!pip install -q torch==2.4.1+cu121 torchvision==0.19.1+cu121 torchaudio==2.4.1+cu121 --extra-index-url https://download.pytorch.org/whl/cu121
!pip install -U datasets huggingface_hub fsspec
!pip install --upgrade openai
!pip install langchain-openai
!pip install tiktoken
!pip install langgraph
!pip install langchain
!pip install langchain-openai
!pip install langchain_community

In [ ]:
from huggingface_hub import login

login(
  token="your-api-key", # ADD YOUR TOKEN HERE
  add_to_git_credential=True
)


## 내 데이터 가져오기

In [ ]:
# DPO를 위한 학습데이터 로드하고 선호도 쌍 형식으로 변환하기

In [ ]:
from datasets import load_dataset

# raw a11y 데이터 로드 (SFT 때 쓰던 것)
raw_ds = load_dataset("doodoo77/a11y-error-dataset-kor", split="train")

IMAGE_TAG = "<image>"

# 사용자 프롬프트 템플릿 (원래 user_prompt에 쓰던 내용으로 바꾸면 됨)
user_prompt_template = """\
다음 UI의 접근성 오류를 설명하고, 문제의 원인과 구체적인 개선 방안을 자세히 작성해 주세요.
오류 코드: {error_code}
"""

def build_dpo_example(sample):
    """
    a11y SFT 샘플 1개 -> DPO용 (prompt, chosen, rejected) 1개로 변환.
    *** 이 함수 안의 필드 이름은 꼭 네 데이터 구조에 맞게 수정해줘! ***
    """

    # 1) 이미지 리스트 정규화
    imgs = sample.get("images", [])
    if not isinstance(imgs, list):
        imgs = [imgs]

    # 2) 텍스트 프롬프트 구성
    #    - Qwen2-VL은 text에 <image> 토큰과 images를 같이 넣으면 알아서 align해줌 :contentReference[oaicite:2]{index=2}
    num_images = len(imgs) if imgs is not None else 0
    image_tokens = IMAGE_TAG * max(num_images, 1)  # 최소 한 개는 넣어 줌

    # 여기는 SFT 노트북에서 쓰던 것 참고: sample["output"]["문제점"] 을 에러 코드로 사용
    error_code = sample["output"]["문제점"]

    user_text = user_prompt_template.format(error_code=error_code)
    prompt = image_tokens + "\n" + user_text

    # 3) chosen / rejected 텍스트 만들기
    #    아래는 "정답 텍스트" 컬럼이 있다고 가정한 예시. 실제 컬럼명 맞게 수정 필요.
    #    예: output 안에 "문제점 및 개선방안_텍스트" 같은 필드가 있을 수도 있음.
    chosen = sample["output"].get(
        "문제점 및 개선방안_텍스트",   # 있을 수도 있는 이름
        sample["output"].get("정답", "")  # 없으면 다른 이름 시도
    )

    # 만약 그런 컬럼이 없으면, 우선 임시로 이렇게 만들고 나중에 꼭 교체해줘:
    if not chosen:
        chosen = f"오류 코드 {error_code}에 대해 자세한 설명과 개선 방안을 작성합니다. (TODO: 실제 정답 텍스트로 교체)"

    # rejected는 “덜 좋은 답변”. 아래는 그냥 대충 나쁜 답을 만드는 예시라,
    # 실제로는 사람이 만든 안 좋은 답/모델의 초기 답 등을 넣어주는 게 좋음.
    rejected = f"오류 코드 {error_code}에 대한 설명이 매우 부족한 답변입니다. (TODO: 실제 나쁜 예시로 교체)"

    return {
        "images": imgs,
        "prompt": prompt,
        "chosen": chosen,
        "rejected": rejected,
    }

dpo_dataset = raw_ds.map(build_dpo_example, remove_columns=raw_ds.column_names)
dpo_dataset[0]


## Model 및 Processor 선언하기

In [ ]:
import torch
from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig

# Hugging Face model id
model_id = "Qwen/Qwen2.5-VL-7B-Instruct"

# BitsAndBytesConfig int-4 config
"""
Qlora를 위해 4비트 양자화를 진행함.

1. 기존 VLM의 파리미터 값을 4비트로 양자화함. (즉, 모든 파라미터 값들을 16가지 값들중 하나로 표현한다는 것임).
   근데 원래 값으로 정규분포를 사용해 복원도 가능함. 이게 놀라운 것임
2. 학습데이터를 기반으로 LoRA adapter 내의 있는 파라미터를 학습함. 이때는 정밀한 loss 계산을 위해 16비트로 표현한 값을 사용함
3. 학습된 LoRA adapter는 full-precision 상태로 저장되고 추론할 때 사용됨
"""
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load model and tokenizer
"""
AutoModelForVision2Seq는 VLM을 로드하기 위한 Hugging Face Transformers의 클래스임
"""
model = AutoModelForVision2Seq.from_pretrained(
    model_id,
    device_map="auto",
    # attn_implementation="flash_attention_2", # not supported for training
    torch_dtype=torch.bfloat16,
    quantization_config=bnb_config
)

"""
AutoProcessor은 이미지, 텍스트 입력 등 다양한 멀티모달 입력에 대해 전처리 및 후처리를 진행하는 클래스임
"""
processor = AutoProcessor.from_pretrained(model_id)

## Lora 하이퍼파라미터 설정할기

In [ ]:
# Lora parameter
# Lora paramter 설정 시 참고할만한 블로그
# https://magazine.sebastianraschka.com/p/practical-tips-for-finetuning-llms
# https://www.youtube.com/watch?v=t1caDsMzWBk

from peft import LoraConfig

"""

- lora_alpha는 학습률로 클 수록 overfitting 되기 쉬우니, 데이터셋이 다양한할때 사용함
- r는 adapter의 중간 차원의 크기임. 보통 lora_alpha의 절반을 사용하는게 좋음 - link
- target_modules -> 즉 lora를 적용한 layer는 attention layer(q_proj, v_proj, k_proj, o_proj)와
MLP layer(gate_proj, up_proj, down_proj) 모두에 적용하는게 좋음

"""

peft_config = LoraConfig(
    lora_alpha = 16, # alpha값은 보통 rank, r값의 2배로 설정함. 근데 qlora에서는 16/64=1/4로 설정함
                     # alpha/rank: scale multiplier -> lora 파라미터 변화량에 곱해져 학습속도에 영향을 줌
                     # 즉, alpha값을 고정하고 learning rate를 조정하는 방향을 추천함

    lora_dropout = 0.1, # overfitting을 막기 위해 파라미터의 일정 비율로 랜덤하게 가중치를 0으로 만듦
                        # 7b-13b에서는 0.1/ 33b-65b에서는 0.05

    r = 8, # rank 값이 클수록 더 복잡한 작업을 수행할 수 있음
            # LoRA의 rank는 model의 원래 weight matrix를 근사하는 데 사용하는 두 작은 행렬의 중간 차원임
            # 즉, 크기 의 두 행렬을 곱하여 d x r, r x d의 두 행렬을 곱해 원래 d x d 행렬을 근사할 때 이 중간 r을 rank라고 부름
            # 근데, QLora에서는 이 r값이 8~256 사이에 있으면 큰 영향이 없음
            # QLora에서는 r값을 64로 시작함

    bias = "none",
    target_modules=["q_proj", "v_proj"], # 다 추가하라면서 왜 해당 layer만 추가한거지?
    task_type="CAUSAL_LM",
)


## DPO 하이퍼파라미터 설정하기

In [ ]:
from trl import DPOConfig

dpo_args = DPOConfig(
    output_dir = "qwen2.5-7b-instruct-a11y-dpo",
    num_train_epochs = 5,
    per_device_train_batch_size = 4,   # DPO는 chosen+rejected 둘 다 돌기 때문에 SFT보다 조금 줄이는 게 안전
    gradient_accumulation_steps = 4,
    gradient_checkpointing = True,
    optim = "adamw_torch_fused",
    logging_steps = 10,
    save_strategy = "epoch",
    learning_rate = 2e-4,
    bf16 = True,
    tf32 = True,
    warmup_ratio = 0.03,
    lr_scheduler_type = "constant",
    push_to_hub = True,
    report_to = "tensorboard",
    gradient_checkpointing_kwargs = {"use_reentrant": False},

    # DPO 전용 옵션들
    beta = 0.1,              # ref 모델에서 얼마나 멀어질지 (작을수록 더 공격적으로 튜닝)
    max_prompt_length = 512,
    max_length = 1024,       # prompt + response 전체 길이
)

# VLM에서 필요한 일부 컬럼을 날리지 않도록
dpo_args.remove_unused_columns = False


## Train model

In [ ]:
torch.cuda.empty_cache()

In [ ]:
from trl import DPOTrainer

dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,            # QLoRA + PEFT 쓸 때는 None으로 두면 내부에서 ref policy를 공유해서 씀
    args = dpo_args,
    train_dataset = dpo_dataset,
    peft_config = peft_config,
    processing_class = processor,  # VLM: tokenizer가 아니라 processor 전체를 넘김
)

dpo_trainer.train()
dpo_trainer.save_model(dpo_args.output_dir)